# Stage 1.5 -- Validation & Feature Engineering: Panel B (Stock Monthly)

## Input
`Data/Data_Collection/Final/Stage_1_Initial_Merge/panel_stock_monthly.parquet` (181 columns, ~25K rows, keyed on `(permno, date)`)

## Purpose
Validation and feature engineering of the merged stock monthly panel before cross-sectional aggregation in Stage 2. The notebook transforms raw dollar-denominated columns into stock-comparable ratios, creates analyst revision dynamics and factor change features, and produces a complete factor inventory. All OAP, IBES, and derived factors are prepared for cap-weighted cross-sectional aggregation.

---

## Initial Diagnostics

A diagnostic pass lists all surviving factor columns with NaN rates, min/max ranges, and identifies non-numeric columns before Block 1 begins.

---

## Block 1: Clean Infinites, Drop Redundant Factors

### Step 1: Replace Infinite Values with NaN
Financial ratios (CashProd, CompositeDebtIssuance, grcapx, etc.) can produce +/-inf when denominators are zero. Rather than dropping entire columns, all infinite values are converted to NaN so the downstream NaN-handling pipeline deals with them (cap-weighted mean ignores NaN). Each affected column and its infinite count are reported before and after replacement.

### Step 2: Pre-Drop Diagnostics
- **Near-zero variance check:** scans all factor columns for effectively constant values (std < 1e-8)
- **`rev_eps_divergence` investigation:** confirms this column is usable despite unusual dtype display in earlier diagnostics
- **`numest` vs `rev_numest` duplicate check:** tests whether OAP's `numest` (EPS analyst count) is identical to IBES's `rev_numest` (revenue analyst count). If exact match rate = 100%, `numest` is flagged for dropping.

### Step 3: Execute Drops (7 columns)
- `yyyymm` -- redundant date identifier (already have `date`)
- `RDIPO` -- all zeros (R&D IPO indicator, never applies to established S&P 100 firms)
- `zerotrade1M`, `zerotrade6M`, `zerotrade12M` -- all zeros (mega-caps always trade every day)
- `Illiquidity` -- Amihud illiquidity ratio is approximately zero for all mega-caps (huge dollar volume makes the ratio vanish)
- `numest` -- exact duplicate of `rev_numest` (confirmed 100% match rate)

### Not Dropped (Needed for Block 3)
`month_end_price`, `month_end_cap`, `ptg_mean/median/high/low`, `rev_mean_fy1` -- these price-level columns are retained for normalisation in Block 3, then dropped at the end of Block 3.

---

## Block 2: Complete Factor Inventory (Pre-Engineering)

Every surviving factor is catalogued with column name, source, category, and description. Sources include OAP (~140 factors across valuation, profitability, accruals, investment, leverage, momentum, risk, liquidity, balance sheet, earnings, tax, corporate events, industry, and sentiment categories), IBES Price Targets (11 factors), IBES Recommendations (13 factors), IBES Revenue (11 factors after `numest` duplicate drop), and Derived (1 factor: `implied_return`).

The inventory is validated bidirectionally against the actual data columns and saved as `stock_monthly_descriptions_pre.csv`.

---

## Block 3: Feature Engineering

Five sections executed in order, creating 32 new features and dropping 6 raw price-level columns.

### Helper Functions
- `safe_div(num, denom)` -- returns NaN when denominator is 0 or NaN
- `grp_diff(col, periods)` -- per-stock monthly difference with **date-gap guard**: if two consecutive rows are more than `periods x 35` days apart (e.g., stock exits universe and re-enters months later), the diff is set to NaN instead of computing a multi-month difference labelled as 1-month. This prevents a 4-month gap from being mislabelled as a 1-month change.
- `grp_roll(col, window, func)` -- per-stock rolling statistic with min_periods safety

### Section A: Normalise Price-Level Columns (6 features)

Transforms raw dollar-denominated analyst targets and revenue estimates into stock-comparable ratios:

- `ptg_median_implied` = ptg_median / month_end_price - 1 (complement to `implied_return` which uses ptg_mean)
- `ptg_upside` = ptg_high / month_end_price - 1 (most optimistic analyst as implied return)
- `ptg_downside` = ptg_low / month_end_price - 1 (most pessimistic analyst as implied return)
- `ptg_implied_range` = ptg_upside - ptg_downside (spread between most optimistic and pessimistic, measures analyst uncertainty about fair value)
- `ptg_implied_asymmetry` = (ptg_upside - implied_return) / (implied_return - ptg_downside) (whether analysts see more room to rise than fall)
- `rev_yield` = rev_mean_fy1 * 1000 / month_end_cap (analyst revenue estimate / market cap, an inverse forward price-to-sales ratio). **Unit fix applied:** IBES revenue is in millions of dollars, CRSP market cap (`dlyprc x shrout`) is in thousands of dollars, so the revenue is multiplied by 1000 to align units. Verified with Apple (PERMNO 14593) producing ~0.10-0.15, consistent with expectations.

### Section B: Analyst Revision Dynamics (9 features)

**B1. Revision acceleration (3 features):** measures whether the revision pace is speeding up or slowing down. `ptg_revision_3m` is the total 3-month revision; dividing by 3 gives the average monthly pace. Acceleration = 1-month revision minus the 3-month average pace. Computed for price targets (`ptg_revision_accel`), recommendations (`rec_revision_accel`), and revenue (`rev_revision_accel`).

**B2. Per-stock monthly changes in analyst sentiment (4 features):** uses `grp_diff` with date-gap guard. `implied_return_chg_1m`, `rec_mean_chg_1m`, `ptg_dispersion_chg_1m`, `rec_dispersion_chg_1m`.

**B3. Cross-source alignment (2 features):** tests whether different analyst signals agree.
- `analyst_alignment` = sign(ptg_revision) x sign(-rec_revision). Note: rec scale is 1=Strong Buy, 5=Strong Sell, so a negative rec_revision means analysts upgraded (moved toward buy); the negation aligns the sign convention. +1 = both signals bullish, -1 = contradictory.
- `ptg_rev_alignment` = sign(ptg_revision) x sign(rev_revision_1m). +1 = both positive, -1 = contradictory.

### Section C: Key Factor Dynamics -- Per-Stock Monthly Changes (11 features)

Computes 1-month (and sometimes 3-month) per-stock differences for market-based factors that update monthly. Accounting-based OAP factors are deliberately excluded (they update quarterly, so diffs would be zero for 2/3 of months then jump). All diffs use the date-gap guarded `grp_diff`.

- `beta_chg_1m`, `beta_chg_3m` -- systematic risk shift
- `idiovol_chg_1m` -- idiosyncratic volatility shift
- `short_interest_chg_1m` -- short interest dynamics
- `realvol_chg_1m` -- realised volatility shift
- `mom12m_chg_1m`, `mom6m_chg_1m` -- momentum acceleration
- `delbreadth_chg_1m` -- institutional breadth shift
- `high52_chg_1m` -- 52-week high proximity shift (breakout/breakdown)
- `volumetrend_chg_1m` -- volume trend acceleration
- `earnings_surprise_chg_1m` -- earnings surprise persistence

The date-gap guard is verified by comparing NaN counts between naive `diff()` and the guarded version, confirming additional NaN from gap detection.

### Section D: Smoothed Levels -- 3-Month Rolling Averages (6 features)

3-month rolling means for noisy signals to capture persistent levels rather than transient spikes:

- `implied_return_3m_avg` -- smoothed analyst optimism
- `rec_mean_3m_avg` -- smoothed consensus recommendation
- `short_interest_3m_avg` -- sustained short-selling pressure
- `idiovol_3m_avg` -- persistent idiosyncratic risk
- `bidask_3m_avg` -- persistent liquidity level
- `rev_yield_3m_avg` -- smoothed revenue yield

### Section E: Drop Raw Price-Level Columns (6 columns dropped)

Columns that are completely consumed by their normalised versions are dropped:
- `ptg_mean` -- consumed by `implied_return`
- `ptg_median` -- consumed by `ptg_median_implied`
- `ptg_high` -- consumed by `ptg_upside`
- `ptg_low` -- consumed by `ptg_downside`
- `rev_mean_fy1` -- consumed by `rev_yield`
- `month_end_price` -- helper column, all normalisation complete

### Sanity Checks
- `ptg_upside > implied_return` for ~100% of valid rows (high target should always exceed mean target)
- `ptg_downside < implied_return` for ~100% of valid rows
- `rev_yield` median in the range 0.05--0.20 for S&P 100 stocks (confirmed)

---

## Block 4: Final Factor Inventory & Save

Every surviving factor is catalogued in a final inventory DataFrame with column name, source, category, and description. The inventory is validated bidirectionally against the actual data columns. Summary statistics by source and category are printed.

**Final factor count: ~194** (from 181 original columns: 7 dropped in Block 1, 32 new features created in Block 3, 6 raw columns replaced in Block 3)

**Factor breakdown by source:**
- OAP: ~134 factors (valuation, profitability, accruals, investment, leverage, momentum/reversal, risk/volatility, liquidity, balance sheet, earnings, tax, corporate events, industry, sentiment)
- IBES Price Targets: 7 factors (raw dollar columns replaced by normalised versions; kept: numest, dispersion, range, upside_skew, revision, revision_3m, numest_chg)
- IBES Recommendations: 13 factors (all retained as-is; already stock-comparable)
- IBES Revenue: 10 factors (raw rev_mean_fy1 replaced by rev_yield; kept: dispersion, range, revisions, coverage, divergence)
- Derived: 33 factors (implied_return, normalised price-levels, analyst revision dynamics, factor dynamics, smoothed levels)

**Key property after engineering:** no raw dollar-denominated columns remain. All factors are either ratios, percentages, counts, or otherwise stock-comparable, ready for cross-sectional aggregation in Stage 2.

## Outputs
- `Data/Data_Collection/Final/Stage_1_5_Validation_and_Feature_Engineering/panel_stock_monthly_engineered.parquet` -- ~194 factor columns plus `permno`, `date`, `month_end_cap` (weight)
- `Data/Data_Collection/Final/Stage_1_5_Validation_and_Feature_Engineering/stock_monthly_descriptions_pre.csv` -- pre-engineering factor inventory
- `Data/Data_Collection/Final/Stage_1_5_Validation_and_Feature_Engineering/stock_monthly_factor_inventory_final.csv` -- final post-engineering factor inventory with source, category, description

In [11]:
import pandas as pd

panel_b = pd.read_parquet(
    '../../../Data/Data_Collection/Final/Stage_1_Initial_Merge/panel_stock_monthly.parquet'
)
panel_b['date'] = pd.to_datetime(panel_b['date'])

factor_cols = [c for c in panel_b.columns if c not in ['permno', 'date', 'month_end_cap', 'month_end_price']]

print(f"Panel B: {len(panel_b):,} rows × {panel_b.shape[1]} columns")
print(f"Factors: {len(factor_cols)}")

# Check for non-numeric
non_numeric = [c for c in factor_cols if not pd.api.types.is_numeric_dtype(panel_b[c])]
print(f"Non-numeric: {non_numeric}")

# Full list with NaN and range
print(f"\n{'#':<5} {'Column':<40} {'NaN%':>7}  {'Min':>12}  {'Max':>12}")
print("-" * 85)
for i, c in enumerate(factor_cols, 1):
    nan_p = panel_b[c].isna().mean() * 100
    vals = panel_b[c].dropna()
    if pd.api.types.is_numeric_dtype(panel_b[c]) and len(vals) > 0:
        print(f"{i:<5} {c:<40} {nan_p:>6.2f}%  {vals.min():>12.4f}  {vals.max():>12.4f}")
    else:
        print(f"{i:<5} {c:<40} {nan_p:>6.2f}%  {'non-numeric':>12}  {'':>12}")

Panel B: 25,194 rows × 181 columns
Factors: 177
Non-numeric: []

#     Column                                      NaN%           Min           Max
-------------------------------------------------------------------------------------
1     yyyymm                                     0.00%   200401.0000   202412.0000
2     AM                                         2.15%        0.0136     1079.0174
3     AOP                                       22.08%     -358.4042        0.9824
4     AbnormalAccruals                           8.35%       -0.4779        1.5495
5     Accruals                                   1.16%       -0.7970        0.5794
6     AnnouncementReturn                         2.18%       -1.0664        0.3518
7     AssetGrowth                                1.16%       -6.5441        0.5243
8     BMdec                                      1.62%       -2.8778        5.7093
9     BPEBM                                      2.41%  -120694.1111     1474.9485
10    Beta         

In [12]:
# %% [markdown]
# # Stage 1.5 — Validation & Feature Engineering: Panel B (Stock Monthly)
#
# Block 1: Replace infinites with NaN, drop redundant/zero-variance factors
# Block 2: Inventory of surviving factors (TBD)
# Block 3: Feature engineering — normalise price-level cols, derive new (TBD)
# Block 4: Final inventory & save (TBD)
#
# Input:  Data/Data_Collection/Final/Stage_1_Initial_Merge/panel_stock_monthly.parquet
# Output: Data/Data_Collection/Final/Stage_1_5_Validation_and_Feature_Engineering/panel_stock_monthly_engineered.parquet

# %%
import pandas as pd
import numpy as np
from pathlib import Path

IN_PATH = Path('../../../Data/Data_Collection/Final/Stage_1_Initial_Merge/panel_stock_monthly.parquet')
OUT_DIR = Path('../../../Data/Data_Collection/Final/Stage_1_5_Validation_and_Feature_Engineering')
OUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_parquet(IN_PATH)
df['date'] = pd.to_datetime(df['date'])

print(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
n_start = df.shape[1]

# ═══════════════════════════════════════════════════════════════════════════════
# BLOCK 1: CLEAN INFINITES, DROP REDUNDANT FACTORS
# ═══════════════════════════════════════════════════════════════════════════════

# %% [markdown]
# ## Block 1: Clean Infinites, Drop Redundant Factors
#
# Steps:
#   1. Replace all ±inf with NaN globally. Financial ratios often produce
#      inf when the denominator is zero (e.g., growth rates, price-to-X).
#      Converting to NaN preserves the column and lets the downstream
#      NaN-handling deal with it (cwmean ignores NaN).
#
#   2. Drop `yyyymm` — redundant date identifier (we have `date`).
#
#   3. Drop near-zero variance factors — constant for top-100 S&P 500 stocks:
#      - RDIPO: all zeros (R&D IPO indicator, never applies to established firms)
#      - zerotrade1M/6M/12M: all zeros (mega-caps always trade every day)
#      - Illiquidity: Amihud ratio ≈ 0 for mega-caps (huge dollar volume)
#
#   4. Price-level columns (ptg_mean/median/high/low, rev_mean_fy1,
#      month_end_price) are NOT dropped here. They are needed in Block 3
#      to compute normalised versions, then dropped at the end of Block 3.

# %%
print("=" * 90)
print("BLOCK 1: CLEAN INFINITES, DROP REDUNDANT FACTORS")
print("=" * 90)

# ── Step 1: Replace ±inf with NaN globally ───────────────────────────────────
# Financial ratios (CashProd, CompositeDebtIssuance, grcapx, etc.) can
# produce ±inf when denominators are zero. Rather than dropping entire
# columns, convert inf to NaN so the NaN-handling pipeline deals with it.

print("\n--- Step 1: Replace ±inf with NaN ---")

# Count infinites before
numeric_cols = df.select_dtypes(include=[np.number]).columns
inf_before = {}
for c in numeric_cols:
    n_inf = np.isinf(df[c]).sum()
    if n_inf > 0:
        inf_before[c] = int(n_inf)

if inf_before:
    print(f"\n  Columns with infinite values BEFORE cleanup:")
    for c, n in sorted(inf_before.items(), key=lambda x: -x[1]):
        total = df[c].notna().sum()
        print(f"    {c:<35s} {n:>6,d} inf  ({n/total*100:.2f}% of valid values)")
else:
    print(f"  ✓ No infinite values found")

# Replace
df = df.replace([np.inf, -np.inf], np.nan)

# Verify
inf_after = 0
for c in numeric_cols:
    inf_after += np.isinf(df[c]).sum()
print(f"\n  Infinite values after cleanup: {inf_after}")
print(f"  ✓ All ±inf replaced with NaN")

# ── Step 2: Pre-drop diagnostics ────────────────────────────────────────────

print("\n--- Step 2: Pre-drop diagnostics ---")

# Near-zero variance check
print("\n  Near-zero variance factors:")
factor_cols = [c for c in df.columns if c not in ['permno', 'date', 'month_end_cap', 'month_end_price']]
low_var_found = []
for c in factor_cols:
    if pd.api.types.is_numeric_dtype(df[c]):
        vals = df[c].dropna()
        if len(vals) > 100:
            nuniq = vals.nunique()
            std = vals.std()
            # Flag if std is essentially zero (constant column)
            # But be careful: binary variables (nunique=2) can be useful
            if pd.notna(std) and float(std) < 1e-8:
                low_var_found.append((c, nuniq, std, vals.min(), vals.max()))
                print(f"    {c:<35s} nunique={nuniq:>3d}  std={std:.2e}  "
                      f"range=[{vals.min():.6f}, {vals.max():.6f}]")

if not low_var_found:
    print(f"    ✓ No zero-variance factors found")

# Check rev_eps_divergence (showed <NA> in earlier diagnostic)
print(f"\n  rev_eps_divergence investigation:")
if 'rev_eps_divergence' in df.columns:
    col = df['rev_eps_divergence']
    print(f"    dtype: {col.dtype}")
    print(f"    NaN: {col.isna().sum()} ({col.isna().mean()*100:.2f}%)")
    valid = col.dropna()
    if len(valid) > 0:
        print(f"    valid count: {len(valid):,}")
        print(f"    sample: {valid.head(5).tolist()}")
        try:
            print(f"    min: {float(valid.min()):.6f}")
            print(f"    max: {float(valid.max()):.6f}")
            print(f"    → Looks usable, keeping it")
        except Exception as e:
            print(f"    ⚠ Cannot compute stats: {e}")
    else:
        print(f"    → All NaN, will be dropped by NaN threshold later")

# Check numest vs rev_numest
print(f"\n  numest vs rev_numest duplicate check:")
if all(c in df.columns for c in ['numest', 'rev_numest']):
    both_valid = df[['numest', 'rev_numest']].dropna()
    if len(both_valid) > 0:
        is_equal = (both_valid['numest'] == both_valid['rev_numest']).mean()
        corr = both_valid['numest'].corr(both_valid['rev_numest'])
        print(f"    Rows with both valid: {len(both_valid):,}")
        print(f"    Exact match rate: {is_equal*100:.1f}%")
        print(f"    Correlation: {corr:.6f}")
        if is_equal == 1.0:
            print(f"    → EXACT DUPLICATE: will drop numest, keep rev_numest")
        else:
            print(f"    → NOT identical: keeping both (EPS analysts ≠ revenue analysts)")

# ── Step 3: Execute drops ────────────────────────────────────────────────────

print("\n--- Step 3: Execute drops ---")

# 3a. Redundant date identifier
drop_date_id = ['yyyymm']

# 3b. Near-zero variance (confirmed constant for S&P 100)
drop_low_var = [
    'RDIPO',          # All zeros — R&D IPO indicator, never applies to S&P 100
    'zerotrade1M',    # All zeros — mega-caps always have daily trading
    'zerotrade6M',    # All zeros
    'zerotrade12M',   # All zeros
    'Illiquidity',    # Amihud ratio ≈ 0 for all mega-caps (huge dollar volume)
]

# 3c. Exact duplicates (only if confirmed above)
drop_exact_dupe = []
if all(c in df.columns for c in ['numest', 'rev_numest']):
    both_valid = df[['numest', 'rev_numest']].dropna()
    if len(both_valid) > 0 and (both_valid['numest'] == both_valid['rev_numest']).mean() == 1.0:
        drop_exact_dupe = ['numest']  # keep rev_numest (clearer name)

# Combine
all_drops = drop_date_id + drop_low_var + drop_exact_dupe

drops_present = [c for c in all_drops if c in df.columns]
drops_missing = [c for c in all_drops if c not in df.columns]

if drops_missing:
    print(f"\n  ⚠ {len(drops_missing)} columns in drop list not found: {drops_missing}")

df = df.drop(columns=drops_present)

# Report
print(f"\n  Drops by category:")
print(f"    Redundant date ID (yyyymm):   {len([c for c in drop_date_id if c in drops_present]):>3d}")
print(f"    Near-zero variance:           {len([c for c in drop_low_var if c in drops_present]):>3d}")
print(f"    Exact duplicates:             {len([c for c in drop_exact_dupe if c in drops_present]):>3d}")
print(f"    ──────────────────────────────")
print(f"    Total dropped:                {len(drops_present):>3d}")
print(f"\n  Columns: {n_start} → {df.shape[1]} ({n_start - df.shape[1]} dropped)")
print(f"\n  NOT dropped (needed for Block 3 normalisation):")
print(f"    month_end_price — for normalising ptg_median/high/low")
print(f"    month_end_cap   — for normalising rev_mean_fy1")
print(f"    ptg_mean/median/high/low — normalised in Block 3, then dropped")
print(f"    rev_mean_fy1 — normalised in Block 3, then dropped")

# Verify required columns intact
for c in ['permno', 'date', 'month_end_cap', 'month_end_price']:
    assert c in df.columns, f"FATAL: Required column '{c}' was dropped!"
print(f"\n  ✓ Required columns intact (permno, date, month_end_cap, month_end_price)")

# ── Final count ──────────────────────────────────────────────────────────────
remaining = [c for c in df.columns if c not in ['permno', 'date', 'month_end_cap', 'month_end_price']]
remaining_numeric = [c for c in remaining if pd.api.types.is_numeric_dtype(df[c])]
print(f"  Remaining factors: {len(remaining)} ({len(remaining_numeric)} numeric)")

# ═══════════════════════════════════════════════════════════════════════════════
# POST-BLOCK 1: DIAGNOSTIC OUTPUT FOR BLOCK 2
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("SURVIVING COLUMNS FOR BLOCK 2")
print("=" * 90)

remaining = [c for c in df.columns if c not in ['permno', 'date', 'month_end_cap', 'month_end_price']]

print(f"\n  Total surviving factors: {len(remaining)}")
print(f"\n{'#':<5} {'Column':<40} {'NaN%':>7}  {'Min':>14}  {'Max':>14}")
print("-" * 85)
for i, c in enumerate(remaining, 1):
    nan_p = df[c].isna().mean() * 100
    if pd.api.types.is_numeric_dtype(df[c]):
        vals = df[c].dropna()
        if len(vals) > 0:
            try:
                mn = f"{float(vals.min()):.4f}"
                mx = f"{float(vals.max()):.4f}"
            except:
                mn = mx = "error"
        else:
            mn = mx = "all NaN"
    else:
        mn = mx = "non-numeric"
    print(f"{i:<5} {c:<40} {nan_p:>6.2f}%  {mn:>14}  {mx:>14}")

Loaded: 25,194 rows × 181 columns
BLOCK 1: CLEAN INFINITES, DROP REDUNDANT FACTORS

--- Step 1: Replace ±inf with NaN ---

  Columns with infinite values BEFORE cleanup:
    CompositeDebtIssuance                  758 inf  (3.19% of valid values)
    grcapx                                  12 inf  (0.05% of valid values)
    CashProd                                 5 inf  (0.02% of valid values)
    rev_eps_divergence                       4 inf  (0.02% of valid values)

  Infinite values after cleanup: 0
  ✓ All ±inf replaced with NaN

--- Step 2: Pre-drop diagnostics ---

  Near-zero variance factors:
    Illiquidity                         nunique=25049  std=2.63e-11  range=[0.000000, 0.000000]
    RDIPO                               nunique=  1  std=0.00e+00  range=[0.000000, 0.000000]
    zerotrade1M                         nunique=25086  std=9.50e-09  range=[0.000000, 0.000000]

  rev_eps_divergence investigation:
    dtype: Float64
    NaN: 2246 (8.91%)
    valid count: 22,948
  

In [13]:
# %% [markdown]
# ## Block 2: Complete Factor Inventory
#
# Every surviving factor named, described, sourced, and categorised.
# Saved as CSV for reference throughout the pipeline.

# %%
print("=" * 90)
print("BLOCK 2: COMPLETE FACTOR INVENTORY")
print("=" * 90)

inventory = []

def add(col, source, category, description):
    inventory.append({
        'column': col,
        'source': source,
        'category': category,
        'description': description,
    })

# ═══════════════════════════════════════════════════════════════════════════════
# OAP: VALUATION (price-to-fundamentals ratios)
# ═══════════════════════════════════════════════════════════════════════════════

add('AM',           'OAP', 'valuation',    'Assets-to-market: total assets / market equity')
add('BMdec',        'OAP', 'valuation',    'Book-to-market (December): book equity / market equity using Dec ME')
add('BPEBM',        'OAP', 'valuation',    'Book price-to-earnings × book-to-market interaction')
add('CF',           'OAP', 'valuation',    'Cash flow-to-price: operating cash flow / market equity')
add('EBM',          'OAP', 'valuation',    'Earnings × book-to-market interaction')
add('EP',           'OAP', 'valuation',    'Earnings-to-price: net income / market equity')
add('EntMult',      'OAP', 'valuation',    'Enterprise multiple: enterprise value / EBITDA')
add('IntanBM',      'OAP', 'valuation',    'Intangible-adjusted book-to-market')
add('IntanCFP',     'OAP', 'valuation',    'Intangible-adjusted cash flow-to-price')
add('IntanEP',      'OAP', 'valuation',    'Intangible-adjusted earnings-to-price')
add('IntanSP',      'OAP', 'valuation',    'Intangible-adjusted sales-to-price')
add('SP',           'OAP', 'valuation',    'Sales-to-price: total revenue / market equity')
add('cfp',          'OAP', 'valuation',    'Cash flow-to-price (alternative definition)')
add('NetPayoutYield','OAP','valuation',    'Net payout yield: (dividends + repurchases - issuance) / market equity')
add('DivYieldST',   'OAP', 'valuation',    'Short-term dividend yield')
add('EquityDuration','OAP','valuation',    'Equity duration: sensitivity of equity value to discount rate changes')

# ═══════════════════════════════════════════════════════════════════════════════
# OAP: PROFITABILITY & QUALITY
# ═══════════════════════════════════════════════════════════════════════════════

add('CBOperProf',   'OAP', 'profitability', 'Cash-based operating profitability')
add('GP',           'OAP', 'profitability', 'Gross profitability: gross profit / total assets')
add('OPLeverage',   'OAP', 'profitability', 'Operating leverage: fixed costs / total costs')
add('OperProf',     'OAP', 'profitability', 'Operating profitability: operating income / book equity')
add('RoE',          'OAP', 'profitability', 'Return on equity: net income / book equity')
add('roaq',         'OAP', 'profitability', 'Return on assets (quarterly): quarterly income / total assets')

# ═══════════════════════════════════════════════════════════════════════════════
# OAP: ACCRUALS & EARNINGS QUALITY
# ═══════════════════════════════════════════════════════════════════════════════

add('AbnormalAccruals','OAP','earnings_quality','Abnormal accruals from modified Jones model')
add('Accruals',     'OAP', 'earnings_quality', 'Total accruals: change in non-cash working capital / total assets')
add('AOP',          'OAP', 'earnings_quality', 'Asset-based operating profitability anomaly')
add('PctAcc',       'OAP', 'earnings_quality', 'Percent accruals: accruals / absolute earnings')
add('PctTotAcc',    'OAP', 'earnings_quality', 'Percent total accruals: total accruals / average total assets')
add('TotalAccruals','OAP', 'earnings_quality', 'Total accruals scaled by average total assets')

# ═══════════════════════════════════════════════════════════════════════════════
# OAP: INVESTMENT & GROWTH
# ═══════════════════════════════════════════════════════════════════════════════

add('AssetGrowth',  'OAP', 'investment',   'Total asset growth: year-over-year change in total assets')
add('ChInv',        'OAP', 'investment',   'Change in inventory scaled by average total assets')
add('ChInvIA',      'OAP', 'investment',   'Change in inventory (industry-adjusted)')
add('CompEquIss',   'OAP', 'investment',   'Composite equity issuance: change in shares + change in equity')
add('GrLTNOA',      'OAP', 'investment',   'Growth in long-term net operating assets')
add('GrSaleToGrInv','OAP', 'investment',   'Growth in sales relative to growth in inventory')
add('GrSaleToGrOverhead','OAP','investment','Growth in sales relative to growth in overhead')
add('Investment',   'OAP', 'investment',   'Capital investment: change in gross PPE / lagged total assets')
add('InvestPPEInv', 'OAP', 'investment',   'Investment in PPE and inventory')
add('XFIN',        'OAP', 'investment',    'External financing: net equity + net debt issuance / total assets')
add('grcapx',      'OAP', 'investment',    'Growth in capital expenditure (year-over-year)')
add('grcapx3y',    'OAP', 'investment',    'Growth in capital expenditure (3-year)')
add('hire',        'OAP', 'investment',    'Employee growth: change in number of employees')

# ═══════════════════════════════════════════════════════════════════════════════
# OAP: FINANCING & CAPITAL STRUCTURE
# ═══════════════════════════════════════════════════════════════════════════════

add('BookLeverage',          'OAP', 'leverage',  'Book leverage: total debt / total assets')
add('Leverage',              'OAP', 'leverage',  'Market leverage: total debt / (total debt + market equity)')
add('CompositeDebtIssuance', 'OAP', 'leverage',  'Composite debt issuance measure')
add('ConvDebt',              'OAP', 'leverage',  'Convertible debt indicator (-1 = has convertible debt, 0 = none)')
add('DebtIssuance',          'OAP', 'leverage',  'Debt issuance indicator (-1 = issued, 0 = did not)')
add('NetDebtFinance',        'OAP', 'leverage',  'Net debt financing: change in debt / total assets')
add('NetEquityFinance',      'OAP', 'leverage',  'Net equity financing: change in equity / total assets')
add('ShareIss1Y',            'OAP', 'leverage',  'Net share issuance (1-year): change in split-adjusted shares')
add('ShareIss5Y',            'OAP', 'leverage',  'Net share issuance (5-year)')
add('ShareRepurchase',       'OAP', 'leverage',  'Share repurchase indicator (1 = repurchased, 0 = did not)')

# ═══════════════════════════════════════════════════════════════════════════════
# OAP: MOMENTUM & REVERSAL
# ═══════════════════════════════════════════════════════════════════════════════

add('Mom12m',                'OAP', 'momentum',  'Momentum 12-month: cumulative return months t-12 to t-2')
add('Mom6m',                 'OAP', 'momentum',  'Momentum 6-month: cumulative return months t-6 to t-2')
add('Mom12mOffSeason',       'OAP', 'momentum',  'Off-season momentum (12-month): returns in non-same calendar months')
add('MomOffSeason',          'OAP', 'momentum',  'Off-season momentum (short horizon)')
add('MomOffSeason06YrPlus',  'OAP', 'momentum',  'Off-season momentum, 6+ year horizon')
add('MomOffSeason11YrPlus',  'OAP', 'momentum',  'Off-season momentum, 11+ year horizon')
add('MomOffSeason16YrPlus',  'OAP', 'momentum',  'Off-season momentum, 16+ year horizon')
add('MomSeason',             'OAP', 'momentum',  'Seasonal momentum: return in same calendar month last year')
add('MomSeason06YrPlus',     'OAP', 'momentum',  'Seasonal momentum, 6+ year horizon')
add('MomSeason11YrPlus',     'OAP', 'momentum',  'Seasonal momentum, 11+ year horizon')
add('MomSeason16YrPlus',     'OAP', 'momentum',  'Seasonal momentum, 16+ year horizon')
add('MomSeasonShort',        'OAP', 'momentum',  'Short-horizon seasonal momentum')
add('IntMom',                'OAP', 'momentum',  'Intermediate momentum: cumulative return months t-7 to t-12')
add('IndMom',                'OAP', 'momentum',  'Industry momentum: value-weighted industry return')
add('MRreversal',            'OAP', 'reversal',  'Medium-run reversal: return months t-18 to t-7')
add('LRreversal',            'OAP', 'reversal',  'Long-run reversal: return months t-60 to t-13')
add('REV6',                  'OAP', 'reversal',  'Short-term reversal: return in month t-1 (6-month variant)')
add('ResidualMomentum',      'OAP', 'momentum',  'Residual momentum: momentum after removing factor exposures')
add('MomVol',                'OAP', 'momentum',  'Momentum-volatility: momentum conditioned on past volatility')
add('TrendFactor',           'OAP', 'momentum',  'Trend factor: weighted moving average of past returns')

# ═══════════════════════════════════════════════════════════════════════════════
# OAP: RISK & VOLATILITY
# ═══════════════════════════════════════════════════════════════════════════════

add('Beta',          'OAP', 'risk',       'CAPM beta: covariance with market / market variance')
add('BetaFP',        'OAP', 'risk',       'Frazzini-Pedersen beta: leveraged/compressed CAPM beta')
add('BetaLiquidityPS','OAP','risk',       'Pastor-Stambaugh liquidity beta: exposure to liquidity factor')
add('BetaTailRisk',  'OAP', 'risk',       'Tail risk beta: exposure to market tail events')
add('betaVIX',       'OAP', 'risk',       'VIX beta: return sensitivity to VIX changes')
add('IdioVol3F',     'OAP', 'risk',       'Idiosyncratic volatility (3-factor): residual vol from FF3 model')
add('IdioVolAHT',    'OAP', 'risk',       'Idiosyncratic volatility (AHT): Ang-Hodrick-Xing-Zhang method')
add('RealizedVol',   'OAP', 'risk',       'Realised volatility: monthly std of daily returns')
add('ReturnSkew',    'OAP', 'risk',       'Return skewness: third moment of daily returns in month')
add('ReturnSkew3F',  'OAP', 'risk',       'Return skewness (3-factor adjusted)')
add('MaxRet',        'OAP', 'risk',       'Maximum daily return in the month (lottery demand proxy)')
add('VolMkt',        'OAP', 'risk',       'Volatility of market beta: time-variation in beta')
add('VolSD',         'OAP', 'risk',       'Volatility of idiosyncratic volatility')
add('VarCF',         'OAP', 'risk',       'Cash flow volatility: std of operating cash flows')
add('CashProd',      'OAP', 'risk',       'Cash productivity: cash / total assets (cash-is-risky anomaly)')

# ═══════════════════════════════════════════════════════════════════════════════
# OAP: LIQUIDITY & TRADING
# ═══════════════════════════════════════════════════════════════════════════════

add('BidAskSpread',  'OAP', 'liquidity',  'Bid-ask spread: monthly average of daily bid-ask spread')
add('DolVol',        'OAP', 'liquidity',  'Log dollar volume: natural log of monthly dollar trading volume')
add('High52',        'OAP', 'liquidity',  'Nearness to 52-week high: current price / 52-week high')
add('VolumeTrend',   'OAP', 'liquidity',  'Volume trend: slope of daily volume over past month')
add('OptionVolume1', 'OAP', 'liquidity',  'Option volume metric 1: options trading activity')
add('OptionVolume2', 'OAP', 'liquidity',  'Option volume metric 2: options trading activity (alternative)')
add('PriceDelayRsq', 'OAP', 'liquidity',  'Price delay (R²): fraction of return variation due to lagged market')
add('PriceDelaySlope','OAP','liquidity',   'Price delay (slope): sensitivity of return to lagged market return')
add('PriceDelayTstat','OAP','liquidity',   'Price delay (t-stat): significance of lagged market return effect')

# ═══════════════════════════════════════════════════════════════════════════════
# OAP: BALANCE SHEET COMPOSITION
# ═══════════════════════════════════════════════════════════════════════════════

add('Cash',          'OAP', 'balance_sheet', 'Cash holdings: cash and short-term investments / total assets')
add('NOA',           'OAP', 'balance_sheet', 'Net operating assets: (operating assets - operating liabilities) / lagged assets')
add('dNoa',          'OAP', 'balance_sheet', 'Change in net operating assets')
add('ChNNCOA',       'OAP', 'balance_sheet', 'Change in non-current net operating assets')
add('ChNWC',         'OAP', 'balance_sheet', 'Change in net working capital')
add('DelCOA',        'OAP', 'balance_sheet', 'Change in current operating assets')
add('DelCOL',        'OAP', 'balance_sheet', 'Change in current operating liabilities')
add('DelEqu',        'OAP', 'balance_sheet', 'Change in book equity')
add('DelFINL',       'OAP', 'balance_sheet', 'Change in financial liabilities')
add('DelLTI',        'OAP', 'balance_sheet', 'Change in long-term investments')
add('DelNetFin',     'OAP', 'balance_sheet', 'Change in net financial assets')

# ═══════════════════════════════════════════════════════════════════════════════
# OAP: EARNINGS & ANALYST FORECASTS
# ═══════════════════════════════════════════════════════════════════════════════

add('AnnouncementReturn','OAP','earnings', 'Earnings announcement abnormal return')
add('EarningsSurprise',  'OAP','earnings', 'Standardised unexpected earnings (SUE)')
add('NumEarnIncrease',   'OAP','earnings', 'Number of consecutive quarterly earnings increases')
add('PredictedFE',       'OAP','earnings', 'Predicted forecast error from cross-sectional model')
add('RevenueSurprise',   'OAP','earnings', 'Revenue surprise: actual vs consensus revenue estimate')
add('fgr5yrLag',         'OAP','earnings', 'Analyst long-term growth forecast (5-year, lagged)')

# ═══════════════════════════════════════════════════════════════════════════════
# OAP: TAX & CORPORATE EVENTS
# ═══════════════════════════════════════════════════════════════════════════════

add('ChTax',         'OAP', 'tax',        'Change in tax expense scaled by total assets')
add('Tax',           'OAP', 'tax',        'Taxable income-to-book income ratio')
add('ExclExp',       'OAP', 'tax',        'Excluded expenses: special items / total assets')
add('ChEQ',          'OAP', 'corporate_events', 'Change in common equity')
add('ChAssetTurnover','OAP','corporate_events',  'Change in asset turnover: ΔSales/Assets')
add('CredRatDG',     'OAP', 'corporate_events', 'Credit rating downgrade indicator (-1 = downgraded)')
add('DivInit',       'OAP', 'corporate_events', 'Dividend initiation indicator (1 = initiated)')
add('DivOmit',       'OAP', 'corporate_events', 'Dividend omission indicator (-1 = omitted)')
add('DivSeason',     'OAP', 'corporate_events', 'Seasonal dividend indicator')
add('ExchSwitch',    'OAP', 'corporate_events', 'Exchange switch indicator (-1 = switched)')
add('IndIPO',        'OAP', 'corporate_events', 'Industry IPO indicator')
add('Spinoff',       'OAP', 'corporate_events', 'Spinoff indicator (1 = spinoff)')

# ═══════════════════════════════════════════════════════════════════════════════
# OAP: INDUSTRY & DIVERSIFICATION
# ═══════════════════════════════════════════════════════════════════════════════

add('Herf',          'OAP', 'industry',   'Herfindahl index: industry sales concentration (negative = concentrated)')
add('HerfAsset',     'OAP', 'industry',   'Herfindahl index based on assets')
add('HerfBE',        'OAP', 'industry',   'Herfindahl index based on book equity')
add('MeanRankRevGrowth','OAP','industry',  'Mean percentile rank of revenue growth across segments')
add('FirmAge',       'OAP', 'industry',   'Firm age: negative months since first CRSP listing')

# ═══════════════════════════════════════════════════════════════════════════════
# OAP: MISCELLANEOUS / OTHER SIGNALS
# ═══════════════════════════════════════════════════════════════════════════════

add('CoskewACX',     'OAP', 'risk',       'Coskewness (ACX): coskewness with market squared returns')
add('Coskewness',    'OAP', 'risk',       'Coskewness: coskewness with market returns')
add('DelBreadth',    'OAP', 'sentiment',  'Change in breadth of institutional ownership')
add('RDS',           'OAP', 'other',      'R&D-to-sales: research & development / total revenue')
add('ShortInterest', 'OAP', 'sentiment',  'Short interest: shares sold short / shares outstanding')

# ═══════════════════════════════════════════════════════════════════════════════
# IBES: PRICE TARGETS (11 factors)
# ═══════════════════════════════════════════════════════════════════════════════

add('ptg_mean',        'IBES_PT', 'price_level', 'Consensus mean analyst price target (raw dollars — normalise in Block 3)')
add('ptg_median',      'IBES_PT', 'price_level', 'Consensus median analyst price target (raw dollars)')
add('ptg_high',        'IBES_PT', 'price_level', 'Highest analyst price target (raw dollars)')
add('ptg_low',         'IBES_PT', 'price_level', 'Lowest analyst price target (raw dollars)')
add('ptg_numest',      'IBES_PT', 'analyst_coverage', 'Number of analysts providing price targets')
add('ptg_dispersion',  'IBES_PT', 'analyst_disagreement', 'Price target dispersion: std / mean of analyst targets')
add('ptg_range',       'IBES_PT', 'analyst_disagreement', 'Price target range: (high - low) / mean')
add('ptg_upside_skew', 'IBES_PT', 'analyst_disagreement', 'Upside skew: (high - median) / (median - low)')
add('ptg_revision',    'IBES_PT', 'analyst_revision', 'Price target revision: 1-month % change in consensus mean')
add('ptg_revision_3m', 'IBES_PT', 'analyst_revision', 'Price target revision: 3-month % change in consensus mean')
add('ptg_numest_chg',  'IBES_PT', 'analyst_coverage', 'Change in number of analysts providing price targets')

# ═══════════════════════════════════════════════════════════════════════════════
# IBES: RECOMMENDATIONS (13 factors)
# ═══════════════════════════════════════════════════════════════════════════════

add('rec_mean',            'IBES_REC', 'analyst_sentiment', 'Mean recommendation: 1=Strong Buy, 5=Strong Sell')
add('rec_median',          'IBES_REC', 'analyst_sentiment', 'Median recommendation: 1=Strong Buy, 5=Strong Sell')
add('rec_numrec',          'IBES_REC', 'analyst_coverage',  'Number of analysts providing recommendations')
add('rec_buy_pct',         'IBES_REC', 'analyst_sentiment', 'Percentage of Buy/Strong Buy recommendations')
add('rec_sell_pct',        'IBES_REC', 'analyst_sentiment', 'Percentage of Sell/Strong Sell recommendations')
add('rec_buy_sell_spread', 'IBES_REC', 'analyst_sentiment', 'Buy% minus Sell% (breadth of bullishness)')
add('rec_dispersion',      'IBES_REC', 'analyst_disagreement', 'Std deviation of analyst recommendations')
add('rec_revision',        'IBES_REC', 'analyst_revision', 'Recommendation revision: 1-month change in mean')
add('rec_revision_3m',     'IBES_REC', 'analyst_revision', 'Recommendation revision: 3-month change in mean')
add('rec_upgrades',        'IBES_REC', 'analyst_revision', 'Number of analysts upgrading recommendation')
add('rec_downgrades',      'IBES_REC', 'analyst_revision', 'Number of analysts downgrading recommendation')
add('rec_changes',         'IBES_REC', 'analyst_revision', 'Total number of recommendation changes')
add('rec_breadth',         'IBES_REC', 'analyst_revision', 'Recommendation breadth: (upgrades - downgrades) / total')

# ═══════════════════════════════════════════════════════════════════════════════
# IBES: REVENUE ESTIMATES (11 factors, after dropping numest duplicate)
# ═══════════════════════════════════════════════════════════════════════════════

add('rev_mean_fy1',        'IBES_REV', 'price_level',         'Mean FY1 revenue estimate (raw dollars — normalise in Block 3)')
add('rev_dispersion',      'IBES_REV', 'analyst_disagreement', 'Revenue estimate dispersion: std / mean')
add('rev_range',           'IBES_REV', 'analyst_disagreement', 'Revenue estimate range: (high - low) / mean')
add('rev_revision_1m',     'IBES_REV', 'analyst_revision',     'Revenue revision: 1-month % change in mean FY1 estimate')
add('rev_revision_3m',     'IBES_REV', 'analyst_revision',     'Revenue revision: 3-month % change in mean FY1 estimate')
add('rev_numest',          'IBES_REV', 'analyst_coverage',     'Number of analysts providing revenue estimates')
add('rev_numest_chg',      'IBES_REV', 'analyst_coverage',     'Change in number of revenue analysts')
add('rev_fy2_revision_1m', 'IBES_REV', 'analyst_revision',     'FY2 revenue revision: 1-month % change in mean FY2 estimate')
add('rev_fy2_dispersion',  'IBES_REV', 'analyst_disagreement', 'FY2 revenue estimate dispersion: std / mean')
add('rev_q_dispersion',    'IBES_REV', 'analyst_disagreement', 'Quarterly revenue estimate dispersion')
add('rev_eps_divergence',  'IBES_REV', 'analyst_disagreement', 'Revenue-EPS divergence: difference in revision direction')

# ═══════════════════════════════════════════════════════════════════════════════
# DERIVED (1 factor)
# ═══════════════════════════════════════════════════════════════════════════════

add('implied_return', 'Derived', 'valuation', 'Implied return: consensus ptg_mean / month_end_price - 1')

# ═══════════════════════════════════════════════════════════════════════════════
# BUILD AND VALIDATE
# ═══════════════════════════════════════════════════════════════════════════════

# %%
inv = pd.DataFrame(inventory)

remaining_factors = [c for c in df.columns if c not in ['permno', 'date', 'month_end_cap', 'month_end_price']]
catalogued = set(inv['column'])

in_data_not_catalogued = [c for c in remaining_factors if c not in catalogued]
in_catalogue_not_data = [c for c in catalogued if c not in remaining_factors]

print(f"\n  Factors in data:       {len(remaining_factors)}")
print(f"  Factors catalogued:    {len(inv)}")

if in_data_not_catalogued:
    print(f"\n  ✗ IN DATA but NOT catalogued ({len(in_data_not_catalogued)}):")
    for c in in_data_not_catalogued:
        print(f"    {c}")
else:
    print(f"  ✓ Every factor in data is catalogued")

if in_catalogue_not_data:
    print(f"\n  ✗ In catalogue but NOT in data ({len(in_catalogue_not_data)}):")
    for c in in_catalogue_not_data:
        print(f"    {c}")
else:
    print(f"  ✓ Every catalogued factor exists in data")

# Summaries
print(f"\n  By source:")
print(inv['source'].value_counts().to_string())

print(f"\n  By category:")
print(inv['category'].value_counts().to_string())

# ═══════════════════════════════════════════════════════════════════════════════
# SAVE INVENTORY
# ═══════════════════════════════════════════════════════════════════════════════

# %%
csv_path = OUT_DIR / 'stock_monthly_descriptions_pre.csv'
csv_string = inv.to_csv(index=False)
with open(csv_path, 'w', encoding='utf-8') as f:
    f.write(csv_string)
print(f"\n  ✓ Saved: {csv_path}")
print(f"    {len(inv)} factors")

# Full table
print(f"\n  Complete inventory:")
print(f"  {'#':<4s} {'Column':<35s} {'Source':<10s} {'Category':<25s} Description")
print("  " + "-" * 120)
for i, row in inv.iterrows():
    print(f"  {i+1:<4d} {row['column']:<35s} {row['source']:<10s} "
          f"{row['category']:<25s} {row['description']}")

BLOCK 2: COMPLETE FACTOR INVENTORY

  Factors in data:       170
  Factors catalogued:    170
  ✓ Every factor in data is catalogued
  ✓ Every catalogued factor exists in data

  By source:
source
OAP         134
IBES_REC     13
IBES_PT      11
IBES_REV     11
Derived       1

  By category:
category
valuation               17
momentum                17
risk                    17
investment              13
analyst_revision        11
balance_sheet           11
leverage                10
corporate_events         9
analyst_disagreement     9
liquidity                9
earnings                 6
profitability            6
earnings_quality         6
industry                 5
price_level              5
analyst_coverage         5
analyst_sentiment        5
tax                      3
reversal                 3
sentiment                2
other                    1

  ✓ Saved: ..\..\..\Data\Data_Collection\Final\Stage_1_5_Validation_and_Feature_Engineering\stock_monthly_descriptions_pre.csv
   

In [14]:
# %% [markdown]
# ## Block 3: Feature Engineering
#
# Five sections:
#   A. Normalise price-level columns (ptg targets → implied returns, revenue → yield)
#   B. Analyst revision dynamics (acceleration, cross-source alignment)
#   C. Key factor dynamics (per-stock monthly changes in market-based factors)
#   D. Smoothed levels (3-month averages for noisy signals)
#   E. Drop raw price-level columns (only columns that are completely consumed)
#
# KEY FIXES IN THIS VERSION:
#   - grp_diff() guards against date gaps: if two consecutive rows are >35 days
#     apart (e.g., stock exits universe and re-enters months later), the diff
#     is set to NaN instead of computing a multi-month difference labeled as 1m.
#   - rev_yield corrects unit mismatch: IBES revenue is in millions, CRSP
#     market cap is in thousands. Multiply by 1000 to align.

# %%
print("=" * 90)
print("BLOCK 3: FEATURE ENGINEERING")
print("=" * 90)

n_before = df.shape[1]
new_features = []

# Ensure sorted for per-stock operations
df = df.sort_values(['permno', 'date']).reset_index(drop=True)

# ── Helpers ──────────────────────────────────────────────────────────────────

def safe_div(num, denom):
    """Safe division: returns NaN when denominator is 0 or NaN."""
    return num / denom.replace(0, np.nan)

def grp_diff(col, periods=1):
    """Per-stock monthly difference with date-gap guard.
    
    If two consecutive rows are more than periods × 35 days apart
    (e.g., stock exits universe and re-enters later), the diff is NaN.
    This prevents a 4-month gap from being labeled as a 1-month change.
    """
    diff_vals = df.groupby('permno')[col].transform(lambda x: x.diff(periods))
    date_diff = df.groupby('permno')['date'].transform(lambda x: x.diff(periods))
    max_days = periods * 35  # 35 days per month allows for 31-day months
    return diff_vals.where(date_diff.dt.days <= max_days, np.nan)

def grp_roll(col, window, func='mean'):
    """Per-stock rolling stat with min_periods safety."""
    min_p = max(window // 2, 1)
    return df.groupby('permno')[col].transform(
        lambda x: getattr(x.rolling(window, min_periods=min_p), func)()
    )

# ═══════════════════════════════════════════════════════════════════════════════
# A. NORMALISE PRICE-LEVEL COLUMNS
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n--- A. Normalise price-level columns ---\n")

a_start = len(new_features)

# Median implied return (complement to implied_return which uses ptg_mean)
df['ptg_median_implied'] = safe_div(df['ptg_median'], df['month_end_price']) - 1
new_features.append('ptg_median_implied')

# Upside target: most optimistic analyst as implied return
df['ptg_upside'] = safe_div(df['ptg_high'], df['month_end_price']) - 1
new_features.append('ptg_upside')

# Downside target: most pessimistic analyst as implied return
df['ptg_downside'] = safe_div(df['ptg_low'], df['month_end_price']) - 1
new_features.append('ptg_downside')

# Implied range: spread between most optimistic and pessimistic analyst
# Large = high analyst uncertainty about fair value
df['ptg_implied_range'] = df['ptg_upside'] - df['ptg_downside']
new_features.append('ptg_implied_range')

# Implied asymmetry: is the upside bigger than the downside?
# Positive = analysts see more room to rise than fall
# Note: can produce outliers when denominator is small — handled by
# winsorisation at 1st/99th percentile in Step 2 aggregation
df['ptg_implied_asymmetry'] = safe_div(
    df['ptg_upside'] - df['implied_return'],
    df['implied_return'] - df['ptg_downside']
)
new_features.append('ptg_implied_asymmetry')

# Revenue yield: analyst revenue estimate / market cap
# UNIT FIX: IBES rev_mean_fy1 is in MILLIONS of dollars.
# CRSP month_end_cap = dlyprc × shrout, where shrout is in THOUSANDS of shares.
# So month_end_cap is in units of (dollars × thousands) = thousands of dollars.
# To get a correct ratio: (millions × 1000) / thousands = dimensionless ratio.
df['rev_yield'] = safe_div(df['rev_mean_fy1'] * 1000, df['month_end_cap'])
new_features.append('rev_yield')

for f in new_features[a_start:]:
    v = df[f].dropna()
    print(f"  {f:<30s} median={v.median():.4f}  range=[{v.min():.4f}, {v.max():.4f}]")

# Verify rev_yield units: Apple (PERMNO 14593) should show ~0.10
apple_check = df[df['permno'] == 14593].tail(1)
if len(apple_check) > 0:
    ry = apple_check['rev_yield'].iloc[0]
    print(f"\n  Unit check — Apple rev_yield: {ry:.4f} (should be ~0.10-0.15)")
    if ry < 0.01 or ry > 1.0:
        print(f"  ⚠ UNIT MISMATCH STILL PRESENT — investigate!")
    else:
        print(f"  ✓ Revenue yield units correct")

print(f"\n  Section A: {len(new_features) - a_start} normalised features")

# ═══════════════════════════════════════════════════════════════════════════════
# B. ANALYST REVISION DYNAMICS
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n--- B. Analyst revision dynamics ---\n")

b_start = len(new_features)

# ── B1. Revision acceleration ────────────────────────────────────────────────
# Is the revision speeding up or slowing down?
# ptg_revision_3m is the TOTAL 3-month revision (not averaged).
# Dividing by 3 gives the average monthly pace over 3 months.
# acceleration > 0 means the most recent month revised faster than the 3m avg.

if all(c in df.columns for c in ['ptg_revision', 'ptg_revision_3m']):
    df['ptg_revision_accel'] = df['ptg_revision'] - (df['ptg_revision_3m'] / 3)
    new_features.append('ptg_revision_accel')

if all(c in df.columns for c in ['rec_revision', 'rec_revision_3m']):
    df['rec_revision_accel'] = df['rec_revision'] - (df['rec_revision_3m'] / 3)
    new_features.append('rec_revision_accel')

if all(c in df.columns for c in ['rev_revision_1m', 'rev_revision_3m']):
    df['rev_revision_accel'] = df['rev_revision_1m'] - (df['rev_revision_3m'] / 3)
    new_features.append('rev_revision_accel')

# ── B2. Per-stock monthly changes in analyst sentiment ───────────────────────
# Uses grp_diff with date-gap guard to prevent multi-month gaps being labeled 1m

if 'implied_return' in df.columns:
    df['implied_return_chg_1m'] = grp_diff('implied_return', 1)
    new_features.append('implied_return_chg_1m')

if 'rec_mean' in df.columns:
    df['rec_mean_chg_1m'] = grp_diff('rec_mean', 1)
    new_features.append('rec_mean_chg_1m')

if 'ptg_dispersion' in df.columns:
    df['ptg_dispersion_chg_1m'] = grp_diff('ptg_dispersion', 1)
    new_features.append('ptg_dispersion_chg_1m')

if 'rec_dispersion' in df.columns:
    df['rec_dispersion_chg_1m'] = grp_diff('rec_dispersion', 1)
    new_features.append('rec_dispersion_chg_1m')

# ── B3. Cross-source alignment ──────────────────────────────────────────────
# Are price target revisions and recommendation revisions aligned?
# Note: rec scale is 1=Strong Buy, 5=Strong Sell, so a NEGATIVE rec_revision
# means analysts upgraded (moved toward buy). We negate for alignment.
# Positive alignment = both signals are bullish.

if all(c in df.columns for c in ['ptg_revision', 'rec_revision']):
    df['analyst_alignment'] = np.sign(df['ptg_revision']) * np.sign(-df['rec_revision'])
    new_features.append('analyst_alignment')

# Revenue revision aligned with price target revision?
if all(c in df.columns for c in ['ptg_revision', 'rev_revision_1m']):
    df['ptg_rev_alignment'] = np.sign(df['ptg_revision']) * np.sign(df['rev_revision_1m'])
    new_features.append('ptg_rev_alignment')

print(f"  Section B: {len(new_features) - b_start} analyst dynamics features")
for f in new_features[b_start:]:
    v = df[f].dropna()
    print(f"    {f:<30s} median={v.median():.4f}  "
          f"range=[{v.min():.4f}, {v.max():.4f}]")

# ═══════════════════════════════════════════════════════════════════════════════
# C. KEY FACTOR DYNAMICS (PER-STOCK MONTHLY CHANGES)
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n--- C. Key factor dynamics (per-stock monthly changes) ---\n")

c_start = len(new_features)

# Only compute dynamics for factors that update monthly (market-based).
# Skip accounting-based OAP factors (they update quarterly — diffs are
# mostly zero for 2/3 months, then jump).
# All diffs use grp_diff with date-gap guard.

# Beta shift: is the stock's systematic risk changing?
if 'Beta' in df.columns:
    df['beta_chg_1m'] = grp_diff('Beta', 1)
    new_features.append('beta_chg_1m')

    df['beta_chg_3m'] = grp_diff('Beta', 3)
    new_features.append('beta_chg_3m')

# Idiosyncratic vol shift
if 'IdioVol3F' in df.columns:
    df['idiovol_chg_1m'] = grp_diff('IdioVol3F', 1)
    new_features.append('idiovol_chg_1m')

# Short interest dynamics
if 'ShortInterest' in df.columns:
    df['short_interest_chg_1m'] = grp_diff('ShortInterest', 1)
    new_features.append('short_interest_chg_1m')

# Realised volatility shift
if 'RealizedVol' in df.columns:
    df['realvol_chg_1m'] = grp_diff('RealizedVol', 1)
    new_features.append('realvol_chg_1m')

# Momentum acceleration
if 'Mom12m' in df.columns:
    df['mom12m_chg_1m'] = grp_diff('Mom12m', 1)
    new_features.append('mom12m_chg_1m')

if 'Mom6m' in df.columns:
    df['mom6m_chg_1m'] = grp_diff('Mom6m', 1)
    new_features.append('mom6m_chg_1m')

# Institutional breadth shift
if 'DelBreadth' in df.columns:
    df['delbreadth_chg_1m'] = grp_diff('DelBreadth', 1)
    new_features.append('delbreadth_chg_1m')

# 52-week high proximity shift (breakout/breakdown)
if 'High52' in df.columns:
    df['high52_chg_1m'] = grp_diff('High52', 1)
    new_features.append('high52_chg_1m')

# Volume trend acceleration
if 'VolumeTrend' in df.columns:
    df['volumetrend_chg_1m'] = grp_diff('VolumeTrend', 1)
    new_features.append('volumetrend_chg_1m')

# Earnings surprise persistence
if 'EarningsSurprise' in df.columns:
    df['earnings_surprise_chg_1m'] = grp_diff('EarningsSurprise', 1)
    new_features.append('earnings_surprise_chg_1m')

print(f"  Section C: {len(new_features) - c_start} factor dynamics features")
for f in new_features[c_start:]:
    v = df[f].dropna()
    print(f"    {f:<30s} median={v.median():.6f}  "
          f"range=[{v.min():.4f}, {v.max():.4f}]")

# ── Verify date-gap guard is working ────────────────────────────────────────
# Count how many diff values were nulled out by the guard
if 'beta_chg_1m' in df.columns:
    naive_nan = df.groupby('permno')['Beta'].transform(lambda x: x.diff(1)).isna().sum()
    guarded_nan = df['beta_chg_1m'].isna().sum()
    gap_nulled = int(guarded_nan) - int(naive_nan)
    print(f"\n  Date-gap guard: {gap_nulled} additional NaN from gap detection "
          f"(rows where consecutive months were >35 days apart)")

# ═══════════════════════════════════════════════════════════════════════════════
# D. SMOOTHED LEVELS (3-MONTH ROLLING AVERAGES)
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n--- D. Smoothed levels (3-month rolling averages) ---\n")

d_start = len(new_features)

# Smoothed implied return
if 'implied_return' in df.columns:
    df['implied_return_3m_avg'] = grp_roll('implied_return', 3)
    new_features.append('implied_return_3m_avg')

# Smoothed recommendation
if 'rec_mean' in df.columns:
    df['rec_mean_3m_avg'] = grp_roll('rec_mean', 3)
    new_features.append('rec_mean_3m_avg')

# Smoothed short interest
if 'ShortInterest' in df.columns:
    df['short_interest_3m_avg'] = grp_roll('ShortInterest', 3)
    new_features.append('short_interest_3m_avg')

# Smoothed idiosyncratic vol
if 'IdioVol3F' in df.columns:
    df['idiovol_3m_avg'] = grp_roll('IdioVol3F', 3)
    new_features.append('idiovol_3m_avg')

# Smoothed bid-ask spread
if 'BidAskSpread' in df.columns:
    df['bidask_3m_avg'] = grp_roll('BidAskSpread', 3)
    new_features.append('bidask_3m_avg')

# Smoothed revenue yield
if 'rev_yield' in df.columns:
    df['rev_yield_3m_avg'] = grp_roll('rev_yield', 3)
    new_features.append('rev_yield_3m_avg')

print(f"  Section D: {len(new_features) - d_start} smoothed features")
for f in new_features[d_start:]:
    v = df[f].dropna()
    print(f"    {f:<30s} median={v.median():.6f}  "
          f"range=[{v.min():.4f}, {v.max():.4f}]")

# ═══════════════════════════════════════════════════════════════════════════════
# E. DROP RAW PRICE-LEVEL COLUMNS
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n--- E. Drop raw price-level columns ---\n")

# Only drop columns that are COMPLETELY consumed by normalised versions.
# Raw dollar amounts are not stock-comparable and would produce
# meaningless cross-sectional statistics.

drop_raw = [
    'ptg_mean',         # Consumed by implied_return
    'ptg_median',       # Consumed by ptg_median_implied
    'ptg_high',         # Consumed by ptg_upside
    'ptg_low',          # Consumed by ptg_downside
    'rev_mean_fy1',     # Consumed by rev_yield
    'month_end_price',  # Helper column, all normalisation complete
]

drops_present = [c for c in drop_raw if c in df.columns]
df = df.drop(columns=drops_present)

print(f"  Dropped {len(drops_present)} raw price-level columns:")
for c in drops_present:
    print(f"    {c}")

# ═══════════════════════════════════════════════════════════════════════════════
# SUMMARY
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("BLOCK 3: SUMMARY")
print("=" * 90)

remaining = [c for c in df.columns if c not in ['permno', 'date', 'month_end_cap']]
remaining_numeric = [c for c in remaining if pd.api.types.is_numeric_dtype(df[c])]

total_a = sum(1 for f in new_features if new_features.index(f) < b_start)
total_b = sum(1 for f in new_features if b_start <= new_features.index(f) < c_start)
total_c = sum(1 for f in new_features if c_start <= new_features.index(f) < d_start)
total_d = sum(1 for f in new_features if new_features.index(f) >= d_start)

print(f"\n  New features by section:")
print(f"    A. Normalised price-levels:    {total_a}")
print(f"    B. Analyst revision dynamics:  {total_b}")
print(f"    C. Factor dynamics (monthly):  {total_c}")
print(f"    D. Smoothed levels (3m avg):   {total_d}")
print(f"    ──────────────────────────────")
print(f"    Total new features:            {len(new_features)}")
print(f"\n  Raw columns dropped:             {len(drops_present)}")
print(f"  Net column change:               {n_before} → {df.shape[1]} columns")
print(f"  Surviving factors:               {len(remaining)} ({len(remaining_numeric)} numeric)")

# Verify required columns intact
for c in ['permno', 'date', 'month_end_cap']:
    assert c in df.columns, f"FATAL: Required column '{c}' was dropped!"
print(f"\n  ✓ Required columns intact")

# Verify all new features exist
missing_new = [f for f in new_features if f not in df.columns]
if missing_new:
    print(f"  ✗ Missing new features: {missing_new}")
else:
    print(f"  ✓ All {len(new_features)} new features present")

# Verify no raw price-level columns remain
remaining_raw = [c for c in df.columns if c in drop_raw]
if remaining_raw:
    print(f"  ⚠ Raw price-level columns still present: {remaining_raw}")
else:
    print(f"  ✓ All raw price-level columns removed")

# NaN check on new features
print(f"\n  New feature NaN rates:")
for f in new_features:
    nan_pct = df[f].isna().mean() * 100
    print(f"    {f:<30s} {nan_pct:>5.2f}%")

# Sanity checks
print(f"\n  Sanity checks:")
if 'ptg_upside' in df.columns and 'implied_return' in df.columns:
    valid = df[['ptg_upside', 'implied_return']].dropna()
    pct = (valid['ptg_upside'] > valid['implied_return']).mean() * 100
    print(f"    ptg_upside > implied_return: {pct:.1f}% (should be ~100%)")

if 'ptg_downside' in df.columns and 'implied_return' in df.columns:
    valid = df[['ptg_downside', 'implied_return']].dropna()
    pct = (valid['ptg_downside'] < valid['implied_return']).mean() * 100
    print(f"    ptg_downside < implied_return: {pct:.1f}% (should be ~100%)")

if 'rev_yield' in df.columns:
    med = df['rev_yield'].median()
    print(f"    rev_yield median: {med:.4f} (should be ~0.05-0.20 for S&P 100)")
    if med < 0.01 or med > 1.0:
        print(f"    ⚠ UNIT MISMATCH — rev_yield outside expected range!")
    else:
        print(f"    ✓ Revenue yield units correct")

# Complete list of new features
print(f"\n  Complete list of {len(new_features)} new features:")
for i, f in enumerate(new_features, 1):
    print(f"    {i:>3d}. {f}")

BLOCK 3: FEATURE ENGINEERING

--- A. Normalise price-level columns ---

  ptg_median_implied             median=0.1102  range=[-0.9813, 323.8408]
  ptg_upside                     median=0.2893  range=[-0.9768, 788.8089]
  ptg_downside                   median=-0.1324  range=[-0.9940, 29.9902]
  ptg_implied_range              median=0.4260  range=[0.0000, 770.7006]
  ptg_implied_asymmetry          median=0.8605  range=[0.1554, 8.0000]
  rev_yield                      median=0.4022  range=[0.0300, 36.4512]

  Unit check — Apple rev_yield: 0.1100 (should be ~0.10-0.15)
  ✓ Revenue yield units correct

  Section A: 6 normalised features

--- B. Analyst revision dynamics ---

  Section B: 9 analyst dynamics features
    ptg_revision_accel             median=-0.1189  range=[-31.4957, 26.5834]
    rec_revision_accel             median=0.0000  range=[-1.3333, 1.3333]
    rev_revision_accel             median=-0.0193  range=[-48.3518, 54.2180]
    implied_return_chg_1m          median=0.0002  r

In [17]:
# %% [markdown]
# ## Block 4: Final Factor Inventory & Save
#
# Catalogues every surviving factor with description, source, and category.
# Saves inventory CSV and engineered panel parquet.

# %%
print("=" * 90)
print("BLOCK 4: FINAL FACTOR INVENTORY & SAVE")
print("=" * 90)

from pathlib import Path

OUT_DIR = Path('../../../Data/Data_Collection/Final/Stage_1_5_Validation_and_Feature_Engineering')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ═══════════════════════════════════════════════════════════════════════════════
# BUILD FINAL INVENTORY
# ═══════════════════════════════════════════════════════════════════════════════

# %%
final_inventory = []

def add(col, source, category, description):
    final_inventory.append({
        'column': col,
        'source': source,
        'category': category,
        'description': description,
    })

# ─────────────────────────────────────────────────────────────────────────────
# OAP: VALUATION (15 — ptg_mean dropped, consumed by implied_return)
# ─────────────────────────────────────────────────────────────────────────────

add('AM',               'OAP', 'valuation',    'Assets-to-market: total assets / market equity')
add('BMdec',            'OAP', 'valuation',    'Book-to-market (December): book equity / market equity using Dec ME')
add('BPEBM',            'OAP', 'valuation',    'Book price-to-earnings × book-to-market interaction')
add('CF',               'OAP', 'valuation',    'Cash flow-to-price: operating cash flow / market equity')
add('EBM',              'OAP', 'valuation',    'Earnings × book-to-market interaction')
add('EP',               'OAP', 'valuation',    'Earnings-to-price: net income / market equity')
add('EntMult',          'OAP', 'valuation',    'Enterprise multiple: enterprise value / EBITDA')
add('IntanBM',          'OAP', 'valuation',    'Intangible-adjusted book-to-market')
add('IntanCFP',         'OAP', 'valuation',    'Intangible-adjusted cash flow-to-price')
add('IntanEP',          'OAP', 'valuation',    'Intangible-adjusted earnings-to-price')
add('IntanSP',          'OAP', 'valuation',    'Intangible-adjusted sales-to-price')
add('SP',               'OAP', 'valuation',    'Sales-to-price: total revenue / market equity')
add('cfp',              'OAP', 'valuation',    'Cash flow-to-price (alternative definition)')
add('NetPayoutYield',   'OAP', 'valuation',    'Net payout yield: (dividends + repurchases - issuance) / ME')
add('DivYieldST',       'OAP', 'valuation',    'Short-term dividend yield')
add('EquityDuration',   'OAP', 'valuation',    'Equity duration: sensitivity to discount rate changes')

# ─────────────────────────────────────────────────────────────────────────────
# OAP: PROFITABILITY & QUALITY
# ─────────────────────────────────────────────────────────────────────────────

add('CBOperProf',  'OAP', 'profitability',    'Cash-based operating profitability')
add('GP',          'OAP', 'profitability',    'Gross profitability: gross profit / total assets')
add('OPLeverage',  'OAP', 'profitability',    'Operating leverage: fixed costs / total costs')
add('OperProf',    'OAP', 'profitability',    'Operating profitability: operating income / book equity')
add('RoE',         'OAP', 'profitability',    'Return on equity: net income / book equity')
add('roaq',        'OAP', 'profitability',    'Return on assets (quarterly)')

# ─────────────────────────────────────────────────────────────────────────────
# OAP: ACCRUALS & EARNINGS QUALITY
# ─────────────────────────────────────────────────────────────────────────────

add('AbnormalAccruals', 'OAP', 'earnings_quality', 'Abnormal accruals from modified Jones model')
add('Accruals',         'OAP', 'earnings_quality', 'Total accruals / total assets')
add('AOP',              'OAP', 'earnings_quality', 'Asset-based operating profitability anomaly')
add('PctAcc',           'OAP', 'earnings_quality', 'Percent accruals: accruals / absolute earnings')
add('PctTotAcc',        'OAP', 'earnings_quality', 'Percent total accruals / average total assets')
add('TotalAccruals',    'OAP', 'earnings_quality', 'Total accruals scaled by average total assets')

# ─────────────────────────────────────────────────────────────────────────────
# OAP: INVESTMENT & GROWTH
# ─────────────────────────────────────────────────────────────────────────────

add('AssetGrowth',          'OAP', 'investment', 'Total asset growth YoY')
add('ChInv',                'OAP', 'investment', 'Change in inventory / avg total assets')
add('ChInvIA',              'OAP', 'investment', 'Change in inventory (industry-adjusted)')
add('CompEquIss',           'OAP', 'investment', 'Composite equity issuance')
add('GrLTNOA',              'OAP', 'investment', 'Growth in long-term net operating assets')
add('GrSaleToGrInv',        'OAP', 'investment', 'Growth in sales vs growth in inventory')
add('GrSaleToGrOverhead',   'OAP', 'investment', 'Growth in sales vs growth in overhead')
add('Investment',            'OAP', 'investment', 'Capital investment: ΔPPE / lagged assets')
add('InvestPPEInv',          'OAP', 'investment', 'Investment in PPE and inventory')
add('XFIN',                 'OAP', 'investment', 'External financing / total assets')
add('grcapx',               'OAP', 'investment', 'Growth in capital expenditure YoY')
add('grcapx3y',             'OAP', 'investment', 'Growth in capital expenditure 3-year')
add('hire',                 'OAP', 'investment', 'Employee growth rate')

# ─────────────────────────────────────────────────────────────────────────────
# OAP: FINANCING & CAPITAL STRUCTURE
# ─────────────────────────────────────────────────────────────────────────────

add('BookLeverage',          'OAP', 'leverage', 'Book leverage: total debt / total assets')
add('Leverage',              'OAP', 'leverage', 'Market leverage: debt / (debt + ME)')
add('CompositeDebtIssuance', 'OAP', 'leverage', 'Composite debt issuance measure')
add('ConvDebt',              'OAP', 'leverage', 'Convertible debt indicator (-1 = yes)')
add('DebtIssuance',          'OAP', 'leverage', 'Debt issuance indicator (-1 = issued)')
add('NetDebtFinance',        'OAP', 'leverage', 'Net debt financing: Δdebt / total assets')
add('NetEquityFinance',      'OAP', 'leverage', 'Net equity financing: Δequity / total assets')
add('ShareIss1Y',            'OAP', 'leverage', 'Net share issuance 1-year')
add('ShareIss5Y',            'OAP', 'leverage', 'Net share issuance 5-year')
add('ShareRepurchase',       'OAP', 'leverage', 'Share repurchase indicator (1 = yes)')

# ─────────────────────────────────────────────────────────────────────────────
# OAP: MOMENTUM & REVERSAL
# ─────────────────────────────────────────────────────────────────────────────

add('Mom12m',               'OAP', 'momentum', 'Momentum 12m: cum return t-12 to t-2')
add('Mom6m',                'OAP', 'momentum', 'Momentum 6m: cum return t-6 to t-2')
add('Mom12mOffSeason',      'OAP', 'momentum', 'Off-season momentum 12m')
add('MomOffSeason',         'OAP', 'momentum', 'Off-season momentum (short)')
add('MomOffSeason06YrPlus', 'OAP', 'momentum', 'Off-season momentum 6+ year')
add('MomOffSeason11YrPlus', 'OAP', 'momentum', 'Off-season momentum 11+ year')
add('MomOffSeason16YrPlus', 'OAP', 'momentum', 'Off-season momentum 16+ year')
add('MomSeason',            'OAP', 'momentum', 'Seasonal momentum: same calendar month last year')
add('MomSeason06YrPlus',    'OAP', 'momentum', 'Seasonal momentum 6+ year')
add('MomSeason11YrPlus',    'OAP', 'momentum', 'Seasonal momentum 11+ year')
add('MomSeason16YrPlus',    'OAP', 'momentum', 'Seasonal momentum 16+ year')
add('MomSeasonShort',       'OAP', 'momentum', 'Short-horizon seasonal momentum')
add('IntMom',               'OAP', 'momentum', 'Intermediate momentum: t-7 to t-12')
add('IndMom',               'OAP', 'momentum', 'Industry momentum: VW industry return')
add('MRreversal',           'OAP', 'reversal', 'Medium-run reversal: t-18 to t-7')
add('LRreversal',           'OAP', 'reversal', 'Long-run reversal: t-60 to t-13')
add('REV6',                 'OAP', 'reversal', 'Short-term reversal: month t-1')
add('ResidualMomentum',     'OAP', 'momentum', 'Residual momentum (factor-adjusted)')
add('MomVol',               'OAP', 'momentum', 'Momentum conditioned on volatility')
add('TrendFactor',          'OAP', 'momentum', 'Trend factor: weighted MA of past returns')

# ─────────────────────────────────────────────────────────────────────────────
# OAP: RISK & VOLATILITY
# ─────────────────────────────────────────────────────────────────────────────

add('Beta',          'OAP', 'risk', 'CAPM beta')
add('BetaFP',        'OAP', 'risk', 'Frazzini-Pedersen beta')
add('BetaLiquidityPS','OAP','risk', 'Pastor-Stambaugh liquidity beta')
add('BetaTailRisk',  'OAP', 'risk', 'Tail risk beta')
add('betaVIX',       'OAP', 'risk', 'VIX beta: sensitivity to VIX changes')
add('IdioVol3F',     'OAP', 'risk', 'Idiosyncratic volatility (FF3 residual)')
add('IdioVolAHT',    'OAP', 'risk', 'Idiosyncratic volatility (AHT method)')
add('RealizedVol',   'OAP', 'risk', 'Realised volatility: std of daily returns')
add('ReturnSkew',    'OAP', 'risk', 'Return skewness')
add('ReturnSkew3F',  'OAP', 'risk', 'Return skewness (3-factor adjusted)')
add('MaxRet',        'OAP', 'risk', 'Max daily return in month (lottery demand)')
add('VolMkt',        'OAP', 'risk', 'Volatility of market beta')
add('VolSD',         'OAP', 'risk', 'Volatility of idiosyncratic volatility')
add('VarCF',         'OAP', 'risk', 'Cash flow volatility')
add('CashProd',      'OAP', 'risk', 'Cash productivity: cash / assets')
add('CoskewACX',     'OAP', 'risk', 'Coskewness (ACX method)')
add('Coskewness',    'OAP', 'risk', 'Coskewness with market')

# ─────────────────────────────────────────────────────────────────────────────
# OAP: LIQUIDITY & TRADING
# ─────────────────────────────────────────────────────────────────────────────

add('BidAskSpread',     'OAP', 'liquidity', 'Monthly avg bid-ask spread')
add('DolVol',           'OAP', 'liquidity', 'Log dollar volume')
add('High52',           'OAP', 'liquidity', 'Price / 52-week high')
add('VolumeTrend',      'OAP', 'liquidity', 'Volume trend: slope of daily volume')
add('OptionVolume1',    'OAP', 'liquidity', 'Options trading activity metric 1')
add('OptionVolume2',    'OAP', 'liquidity', 'Options trading activity metric 2')
add('PriceDelayRsq',    'OAP', 'liquidity', 'Price delay R²: lagged market return fraction')
add('PriceDelaySlope',  'OAP', 'liquidity', 'Price delay slope')
add('PriceDelayTstat',  'OAP', 'liquidity', 'Price delay t-statistic')

# ─────────────────────────────────────────────────────────────────────────────
# OAP: BALANCE SHEET
# ─────────────────────────────────────────────────────────────────────────────

add('Cash',      'OAP', 'balance_sheet', 'Cash / total assets')
add('NOA',       'OAP', 'balance_sheet', 'Net operating assets / lagged assets')
add('dNoa',      'OAP', 'balance_sheet', 'Change in net operating assets')
add('ChNNCOA',   'OAP', 'balance_sheet', 'Change in non-current net operating assets')
add('ChNWC',     'OAP', 'balance_sheet', 'Change in net working capital')
add('DelCOA',    'OAP', 'balance_sheet', 'Change in current operating assets')
add('DelCOL',    'OAP', 'balance_sheet', 'Change in current operating liabilities')
add('DelEqu',    'OAP', 'balance_sheet', 'Change in book equity')
add('DelFINL',   'OAP', 'balance_sheet', 'Change in financial liabilities')
add('DelLTI',    'OAP', 'balance_sheet', 'Change in long-term investments')
add('DelNetFin', 'OAP', 'balance_sheet', 'Change in net financial assets')

# ─────────────────────────────────────────────────────────────────────────────
# OAP: EARNINGS & FORECASTS
# ─────────────────────────────────────────────────────────────────────────────

add('AnnouncementReturn', 'OAP', 'earnings', 'Earnings announcement abnormal return')
add('EarningsSurprise',   'OAP', 'earnings', 'Standardised unexpected earnings (SUE)')
add('NumEarnIncrease',    'OAP', 'earnings', 'Consecutive quarterly earnings increases')
add('PredictedFE',        'OAP', 'earnings', 'Predicted forecast error')
add('RevenueSurprise',    'OAP', 'earnings', 'Revenue surprise: actual vs consensus')
add('fgr5yrLag',          'OAP', 'earnings', 'Analyst 5-year growth forecast (lagged)')

# ─────────────────────────────────────────────────────────────────────────────
# OAP: TAX & CORPORATE EVENTS
# ─────────────────────────────────────────────────────────────────────────────

add('ChTax',            'OAP', 'tax',              'Change in tax expense / total assets')
add('Tax',              'OAP', 'tax',              'Taxable-to-book income ratio')
add('ExclExp',          'OAP', 'tax',              'Special items / total assets')
add('ChEQ',             'OAP', 'corporate_events', 'Change in common equity')
add('ChAssetTurnover',  'OAP', 'corporate_events', 'Change in asset turnover')
add('CredRatDG',        'OAP', 'corporate_events', 'Credit rating downgrade (-1)')
add('DivInit',          'OAP', 'corporate_events', 'Dividend initiation (1)')
add('DivOmit',          'OAP', 'corporate_events', 'Dividend omission (-1)')
add('DivSeason',        'OAP', 'corporate_events', 'Seasonal dividend indicator')
add('ExchSwitch',       'OAP', 'corporate_events', 'Exchange switch indicator (-1)')
add('IndIPO',           'OAP', 'corporate_events', 'Industry IPO indicator')
add('Spinoff',          'OAP', 'corporate_events', 'Spinoff indicator (1)')

# ─────────────────────────────────────────────────────────────────────────────
# OAP: INDUSTRY & SENTIMENT
# ─────────────────────────────────────────────────────────────────────────────

add('Herf',              'OAP', 'industry',  'Herfindahl: industry sales concentration')
add('HerfAsset',         'OAP', 'industry',  'Herfindahl based on assets')
add('HerfBE',            'OAP', 'industry',  'Herfindahl based on book equity')
add('MeanRankRevGrowth', 'OAP', 'industry',  'Mean rank of revenue growth across segments')
add('FirmAge',           'OAP', 'industry',  'Negative months since first CRSP listing')
add('DelBreadth',        'OAP', 'sentiment', 'Change in institutional ownership breadth')
add('RDS',               'OAP', 'other',     'R&D-to-sales')
add('ShortInterest',     'OAP', 'sentiment', 'Shares sold short / shares outstanding')

# ─────────────────────────────────────────────────────────────────────────────
# IBES: PRICE TARGETS (6 — raw ptg_mean/median/high/low dropped)
# ─────────────────────────────────────────────────────────────────────────────

add('ptg_numest',      'IBES_PT', 'analyst_coverage',     'Number of analysts with price targets')
add('ptg_dispersion',  'IBES_PT', 'analyst_disagreement', 'Price target dispersion: std / mean')
add('ptg_range',       'IBES_PT', 'analyst_disagreement', 'Price target range: (high - low) / mean')
add('ptg_upside_skew', 'IBES_PT', 'analyst_disagreement', 'Upside skew: (high - median) / (median - low)')
add('ptg_revision',    'IBES_PT', 'analyst_revision',     'Price target revision: 1m % change in mean')
add('ptg_revision_3m', 'IBES_PT', 'analyst_revision',     'Price target revision: 3m total % change in mean')
add('ptg_numest_chg',  'IBES_PT', 'analyst_coverage',     'Change in number of price target analysts')

# ─────────────────────────────────────────────────────────────────────────────
# IBES: RECOMMENDATIONS (13)
# ─────────────────────────────────────────────────────────────────────────────

add('rec_mean',            'IBES_REC', 'analyst_sentiment',    'Mean recommendation (1=Strong Buy, 5=Strong Sell)')
add('rec_median',          'IBES_REC', 'analyst_sentiment',    'Median recommendation')
add('rec_numrec',          'IBES_REC', 'analyst_coverage',     'Number of recommendation analysts')
add('rec_buy_pct',         'IBES_REC', 'analyst_sentiment',    '% Buy/Strong Buy recommendations')
add('rec_sell_pct',        'IBES_REC', 'analyst_sentiment',    '% Sell/Strong Sell recommendations')
add('rec_buy_sell_spread', 'IBES_REC', 'analyst_sentiment',    'Buy% minus Sell%')
add('rec_dispersion',      'IBES_REC', 'analyst_disagreement', 'Std of recommendations')
add('rec_revision',        'IBES_REC', 'analyst_revision',     'Recommendation revision: 1m change in mean')
add('rec_revision_3m',     'IBES_REC', 'analyst_revision',     'Recommendation revision: 3m change in mean')
add('rec_upgrades',        'IBES_REC', 'analyst_revision',     'Number of upgrades')
add('rec_downgrades',      'IBES_REC', 'analyst_revision',     'Number of downgrades')
add('rec_changes',         'IBES_REC', 'analyst_revision',     'Total recommendation changes')
add('rec_breadth',         'IBES_REC', 'analyst_revision',     'Breadth: (upgrades - downgrades) / total')

# ─────────────────────────────────────────────────────────────────────────────
# IBES: REVENUE (10 — raw rev_mean_fy1 dropped)
# ─────────────────────────────────────────────────────────────────────────────

add('rev_dispersion',      'IBES_REV', 'analyst_disagreement', 'Revenue estimate dispersion: std / mean')
add('rev_range',           'IBES_REV', 'analyst_disagreement', 'Revenue estimate range: (high - low) / mean')
add('rev_revision_1m',     'IBES_REV', 'analyst_revision',     'Revenue revision 1m: % change in mean FY1')
add('rev_revision_3m',     'IBES_REV', 'analyst_revision',     'Revenue revision 3m: total % change in mean FY1')
add('rev_numest',          'IBES_REV', 'analyst_coverage',     'Number of revenue analysts')
add('rev_numest_chg',      'IBES_REV', 'analyst_coverage',     'Change in number of revenue analysts')
add('rev_fy2_revision_1m', 'IBES_REV', 'analyst_revision',     'FY2 revenue revision 1m')
add('rev_fy2_dispersion',  'IBES_REV', 'analyst_disagreement', 'FY2 revenue estimate dispersion')
add('rev_q_dispersion',    'IBES_REV', 'analyst_disagreement', 'Quarterly revenue estimate dispersion')
add('rev_eps_divergence',  'IBES_REV', 'analyst_disagreement', 'Revenue-EPS revision divergence')

# ─────────────────────────────────────────────────────────────────────────────
# DERIVED: ORIGINAL (1)
# ─────────────────────────────────────────────────────────────────────────────

add('implied_return', 'Derived', 'valuation', 'Implied return: ptg_mean / price - 1')

# ─────────────────────────────────────────────────────────────────────────────
# BLOCK 3 NEW: NORMALISED PRICE-LEVELS (Section A)
# ─────────────────────────────────────────────────────────────────────────────

add('ptg_median_implied',   'Derived', 'valuation',            'Median price target implied return: ptg_median / price - 1')
add('ptg_upside',           'Derived', 'valuation',            'Upside target: ptg_high / price - 1 (max analyst optimism)')
add('ptg_downside',         'Derived', 'valuation',            'Downside target: ptg_low / price - 1 (max analyst pessimism)')
add('ptg_implied_range',    'Derived', 'analyst_disagreement', 'Implied range: ptg_upside - ptg_downside (analyst uncertainty)')
add('ptg_implied_asymmetry','Derived', 'analyst_disagreement', 'Implied asymmetry: (upside - mean) / (mean - downside)')
add('rev_yield',            'Derived', 'valuation',            'Revenue yield: IBES revenue / market cap (inverse forward P/S)')

# ─────────────────────────────────────────────────────────────────────────────
# BLOCK 3 NEW: ANALYST REVISION DYNAMICS (Section B)
# ─────────────────────────────────────────────────────────────────────────────

add('ptg_revision_accel',   'Derived', 'analyst_revision', 'PT revision acceleration: 1m revision vs 3m avg pace')
add('rec_revision_accel',   'Derived', 'analyst_revision', 'Rec revision acceleration: 1m revision vs 3m avg pace')
add('rev_revision_accel',   'Derived', 'analyst_revision', 'Revenue revision acceleration: 1m revision vs 3m avg pace')
add('implied_return_chg_1m','Derived', 'analyst_revision', '1-month change in implied return (per-stock)')
add('rec_mean_chg_1m',      'Derived', 'analyst_revision', '1-month change in mean recommendation (per-stock)')
add('ptg_dispersion_chg_1m','Derived', 'analyst_disagreement', '1-month change in PT dispersion (per-stock)')
add('rec_dispersion_chg_1m','Derived', 'analyst_disagreement', '1-month change in rec dispersion (per-stock)')
add('analyst_alignment',    'Derived', 'analyst_revision', 'PT × Rec alignment: +1 = both bullish, -1 = contradictory')
add('ptg_rev_alignment',    'Derived', 'analyst_revision', 'PT × Revenue alignment: +1 = both positive, -1 = contradictory')

# ─────────────────────────────────────────────────────────────────────────────
# BLOCK 3 NEW: KEY FACTOR DYNAMICS (Section C)
# ─────────────────────────────────────────────────────────────────────────────

add('beta_chg_1m',              'Derived', 'risk_dynamics',     '1-month change in CAPM beta (per-stock)')
add('beta_chg_3m',              'Derived', 'risk_dynamics',     '3-month change in CAPM beta (per-stock)')
add('idiovol_chg_1m',           'Derived', 'risk_dynamics',     '1-month change in idiosyncratic vol (per-stock)')
add('short_interest_chg_1m',    'Derived', 'sentiment_dynamics','1-month change in short interest (per-stock)')
add('realvol_chg_1m',           'Derived', 'risk_dynamics',     '1-month change in realised vol (per-stock)')
add('mom12m_chg_1m',            'Derived', 'momentum_dynamics', '1-month change in 12m momentum (acceleration)')
add('mom6m_chg_1m',             'Derived', 'momentum_dynamics', '1-month change in 6m momentum (acceleration)')
add('delbreadth_chg_1m',        'Derived', 'sentiment_dynamics','1-month change in inst ownership breadth')
add('high52_chg_1m',            'Derived', 'momentum_dynamics', '1-month change in proximity to 52w high')
add('volumetrend_chg_1m',       'Derived', 'liquidity_dynamics','1-month change in volume trend')
add('earnings_surprise_chg_1m', 'Derived', 'earnings_dynamics', '1-month change in earnings surprise')

# ─────────────────────────────────────────────────────────────────────────────
# BLOCK 3 NEW: SMOOTHED LEVELS (Section D)
# ─────────────────────────────────────────────────────────────────────────────

add('implied_return_3m_avg', 'Derived', 'valuation',            '3-month avg implied return (smoothed analyst optimism)')
add('rec_mean_3m_avg',       'Derived', 'analyst_sentiment',    '3-month avg recommendation (smoothed consensus)')
add('short_interest_3m_avg', 'Derived', 'sentiment',            '3-month avg short interest (sustained pressure)')
add('idiovol_3m_avg',        'Derived', 'risk',                 '3-month avg idiosyncratic vol (persistent risk)')
add('bidask_3m_avg',         'Derived', 'liquidity',            '3-month avg bid-ask spread (persistent liquidity)')
add('rev_yield_3m_avg',      'Derived', 'valuation',            '3-month avg revenue yield (smoothed)')

# ═══════════════════════════════════════════════════════════════════════════════
# VALIDATE INVENTORY VS ACTUAL DATA
# ═══════════════════════════════════════════════════════════════════════════════

# %%
inv = pd.DataFrame(final_inventory)

remaining_factors = [c for c in df.columns if c not in ['permno', 'date', 'month_end_cap']]
catalogued = set(inv['column'])

in_data_not_catalogued = [c for c in remaining_factors if c not in catalogued]
in_catalogue_not_data = [c for c in catalogued if c not in remaining_factors]

print(f"\n  Factors in data:       {len(remaining_factors)}")
print(f"  Factors catalogued:    {len(inv)}")

if in_data_not_catalogued:
    print(f"\n  ✗ IN DATA but NOT catalogued ({len(in_data_not_catalogued)}):")
    for c in in_data_not_catalogued:
        print(f"    {c}")
else:
    print(f"  ✓ Every factor in data is catalogued")

if in_catalogue_not_data:
    print(f"\n  ✗ In catalogue but NOT in data ({len(in_catalogue_not_data)}):")
    for c in in_catalogue_not_data:
        print(f"    {c}")
else:
    print(f"  ✓ Every catalogued factor exists in data")

# Summaries
print(f"\n  By source:")
print(inv['source'].value_counts().to_string())

print(f"\n  By category:")
print(inv['category'].value_counts().to_string())

# ═══════════════════════════════════════════════════════════════════════════════
# SAVE INVENTORY CSV
# ═══════════════════════════════════════════════════════════════════════════════

# %%
csv_path = OUT_DIR / 'stock_monthly_factor_inventory_final.csv'
csv_string = inv.to_csv(index=False)
with open(csv_path, 'w', encoding='utf-8') as f:
    f.write(csv_string)
print(f"\n  ✓ Inventory saved: {csv_path}")
print(f"    {len(inv)} factors")

# ═══════════════════════════════════════════════════════════════════════════════
# SAVE ENGINEERED PANEL TO PARQUET
# ═══════════════════════════════════════════════════════════════════════════════

# %%
df = df.sort_values(['permno', 'date']).reset_index(drop=True)

parquet_path = OUT_DIR / 'panel_stock_monthly_engineered.parquet'
df.to_parquet(parquet_path, index=False, engine='pyarrow')

file_size = parquet_path.stat().st_size
print(f"\n  ✓ Panel saved: {parquet_path}")
print(f"    {len(df):,} rows × {df.shape[1]} columns")
print(f"    ID: permno, date")
print(f"    Weight: month_end_cap")
print(f"    Factors: {len(remaining_factors)}")
print(f"    Size: {file_size / 1e6:.1f} MB")

# ═══════════════════════════════════════════════════════════════════════════════
# FINAL SUMMARY
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("PANEL B FEATURE ENGINEERING COMPLETE")
print("=" * 90)

print(f"""
  Pipeline: Raw (181 cols) → Block 1 (174) → Block 3 (200) → Final ({df.shape[1]})

  Final panel:
    Rows:       {len(df):,}
    Columns:    {df.shape[1]}
    Factors:    {len(remaining_factors)}
    PERMNOs:    {df['permno'].nunique()}
    Dates:      {df['date'].nunique():,}
    Date range: {df['date'].min().date()} → {df['date'].max().date()}

  Saved to:
    Panel:     {parquet_path}
    Inventory: {csv_path}

  Next step: Stage 2 aggregation reads this panel and computes
  cap-weighted cross-sectional statistics per month, producing
  market-level monthly time series.
""")

BLOCK 4: FINAL FACTOR INVENTORY & SAVE

  Factors in data:       197
  Factors catalogued:    197
  ✓ Every factor in data is catalogued
  ✓ Every catalogued factor exists in data

  By source:
source
OAP         134
Derived      33
IBES_REC     13
IBES_REV     10
IBES_PT       7

  By category:
category
valuation               23
analyst_revision        18
risk                    18
momentum                17
investment              13
analyst_disagreement    13
balance_sheet           11
leverage                10
liquidity               10
corporate_events         9
earnings                 6
profitability            6
earnings_quality         6
analyst_sentiment        6
industry                 5
analyst_coverage         5
risk_dynamics            4
tax                      3
sentiment                3
reversal                 3
momentum_dynamics        3
sentiment_dynamics       2
other                    1
liquidity_dynamics       1
earnings_dynamics        1

  ✓ Inventory save